In [ ]:
import pandas as pd
import numpy as np

# Load Dataset
df = pd.read_csv("tesla_deliveries_dataset_2015_2025.csv")

print("Shape:", df.shape)
print(df.head())

# DATA PREPROCESSING

In [ ]:
# Missing Values
print("\nMissing Values:")
print(df.isnull().sum())

# Create Date Column
df["Date"] = pd.to_datetime(
    df["Year"].astype(str) + "-" +
    df["Month"].astype(str) + "-01"
)

In [ ]:

# Feature Engineering
df["Quarter"] = df["Date"].dt.quarter

df["Production_Delivery_Ratio"] = (
    df["Production_Units"] /
    df["Estimated_Deliveries"]
)

In [ ]:
# Fill any infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Price Distribution
plt.figure(figsize=(8,5))
sns.histplot(df["Avg_Price_USD"], kde=True)
plt.title("Average Price Distribution")
plt.show()

In [ ]:
# Deliveries Trend
plt.figure(figsize=(10,5))
df.groupby("Year")["Estimated_Deliveries"].sum().plot()
plt.title("Tesla Deliveries by Year")
plt.ylabel("Deliveries")
plt.show()

In [ ]:
# Correlation Heatmap
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12,8))
sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Correlation Matrix")
plt.show()

# ML MODELS

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

In [ ]:
# Target
y = df["Avg_Price_USD"]

# Remove target and date
X = df.drop(
    columns=[
        "Avg_Price_USD",
        "Date"
    ]
)

In [ ]:
# Numeric Columns
num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

# Categorical Columns
cat_cols = X.select_dtypes(
    include=["object","str"]
).columns

# Preprocessor

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

# Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# LINEAR MODEL

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_pipeline.fit(X_train, y_train)

lr_preds = lr_pipeline.predict(X_test)

print("\nLinear Regression Results")
print("MAE:", mean_absolute_error(y_test, lr_preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_preds)))
print("R2:", r2_score(y_test, lr_preds))

# LASSO MODEL

In [ ]:
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(alpha=0.1,
                   max_iter=8000))
])

lasso_pipeline.fit(X_train, y_train)

lasso_preds = lasso_pipeline.predict(X_test)

print("\nLasso Regression Results")
print("MAE:", mean_absolute_error(y_test, lasso_preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lasso_preds)))
print("R2:", r2_score(y_test, lasso_preds))

# HYPER TUNNING FOR LASSO

In [ ]:
from sklearn.model_selection import GridSearchCV

lasso_grid = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 10]
}

lasso_search = GridSearchCV(
    lasso_pipeline,
    lasso_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)

print("\nBest Lasso Alpha:")
print(lasso_search.best_params_)

best_lasso = lasso_search.best_estimator_

lasso_preds = best_lasso.predict(X_test)

print("\nTuned Lasso Results")
print("MAE:", mean_absolute_error(y_test, lasso_preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lasso_preds)))
print("R2:", r2_score(y_test, lasso_preds))

# RANDOM FOREST MODEL

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

# Predictions
preds = rf_pipeline.predict(X_test)

# Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)


In [ ]:
print("\nRandom Forest Results")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

# HYPERPARAMATER TUNNING

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_

preds_best = best_model.predict(X_test)

In [ ]:
print("\nTuned Model Results")
print(
    "MAE:",
    mean_absolute_error(y_test, preds_best)
)

print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_test, preds_best))
)

print(
    "R2:",
    r2_score(y_test, preds_best)
)

# FEATURE IMPORTANCE

In [ ]:
feature_names = (
    num_cols.tolist() +
    list(
        best_model.named_steps["preprocessor"]
        .named_transformers_["cat"]
        .named_steps["encoder"]
        .get_feature_names_out(cat_cols)
    )
)

importances = (
    best_model.named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Features")
print(importance_df.head(10))


# TIME SERIES FORECASTING

In [ ]:
monthly = (
    df.groupby("Date")["Estimated_Deliveries"]
    .sum()
    .reset_index()
)

monthly = monthly.sort_values("Date")

monthly.set_index("Date", inplace=True)

from statsmodels.tsa.arima.model import ARIMA


monthly = monthly.asfreq('MS')

model = ARIMA(
    monthly["Estimated_Deliveries"],
    order=(5,1,0)
)

result = model.fit()

forecast = result.forecast(steps=12)

print("\nNext 12 Months Forecast")
print(forecast)

# Forecast Plot
plt.figure(figsize=(12,6))

plt.plot(
    monthly.index,
    monthly["Estimated_Deliveries"],
    label="Historical"
)

future_dates = pd.date_range(
    start=monthly.index.max(),
    periods=13,
    freq="MS"
)[1:]

plt.plot(
    future_dates,
    forecast,
    label="Forecast"
)

plt.legend()
plt.title("Tesla Deliveries Forecast")
plt.show()

# COMPARISON OF ALL MODELS

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Lasso",
        "Random Forest"
    ],
    "MAE": [
        mean_absolute_error(y_test, lr_preds),
        mean_absolute_error(y_test, lasso_preds),
        mean_absolute_error(y_test, preds_best)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, lr_preds)),
        np.sqrt(mean_squared_error(y_test, lasso_preds)),
        np.sqrt(mean_squared_error(y_test, preds_best))
    ],
    "R2": [
        r2_score(y_test, lr_preds),
        r2_score(y_test, lasso_preds),
        r2_score(y_test, preds_best)
    ]
})

print(results.sort_values("R2", ascending=False))